# Live Grep to SQL Chunk Harness

This notebook tests the chunk-search database described in the research notes.

Flow:

1. Compile the grep pattern into the authoritative Python regex.
2. Inspect required literals and approximate trigrams for visibility.
3. Translate the grep/glob request into a PostgreSQL candidate predicate over `kind='chunk'` rows.
4. Fetch chunk candidates with content and group them by owning file path.
5. Run final Python line matching on the chunk content. SQL narrows candidates; Python decides correctness.


In [ ]:
from __future__ import annotations

import os
import re
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from pprint import pprint
from typing import Any

from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "grep_glob research"))

from pushdown_extract import contains_unescaped_anchor, required_regex_literals
from vfs.backends.database import _compile_grep_regex, _escape_like, _regex_flags_for_mode
from vfs.backends.postgres import _python_regex_to_postgres
from vfs.patterns import compile_glob, glob_to_sql_like


## Controls

Set `GROVER_TRGM_DB_URL` if your connection differs from the local default. The SQL always includes the partial-index predicate required by the chunk trigram index:

```sql
kind = 'chunk'
AND content IS NOT NULL
AND deleted_at IS NULL
```


In [2]:
DB_URL = os.getenv("GROVER_TRGM_DB_URL", "postgresql+asyncpg://localhost/pg_trgrm_test_git_repos")
SCHEMA = "public"
TABLE = "vfs_entries"
CHUNK_MARKER = "/__meta__/chunks/"

pattern = "Postgres(FileSystem|Backend)"
case_mode = "sensitive"  # sensitive | insensitive | smart
fixed_strings = False
word_regexp = False
invert_match = False
glob = "**/*.py"

SQL_LIMIT = 5000
DISPLAY_LIMIT = 25
RUN_EXPLAIN = True

engine = create_async_engine(DB_URL, pool_pre_ping=True)


In [3]:
@dataclass(frozen=True)
class CandidatePredicate:
    sql: str
    params: dict[str, Any]
    description: str
    regex: re.Pattern[str]
    required_literals: tuple[str, ...]
    approximate_trigrams: dict[str, tuple[str, ...]]


def _sql_ident(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Unsafe SQL identifier: {name!r}")
    return name


def table_ref(schema: str = SCHEMA, table: str = TABLE) -> str:
    return f"{_sql_ident(schema)}.{_sql_ident(table)}"


def approximate_trigrams(text_value: str) -> tuple[str, ...]:
    """Human-readable trigram sketch; PostgreSQL computes exact pg_trgm keys internally."""
    value = text_value.lower()
    if len(value) < 3:
        return ()
    return tuple(sorted({value[i : i + 3] for i in range(len(value) - 2)}))


def build_content_predicate(
    pattern: str,
    *,
    case_mode: str,
    fixed_strings: bool,
    word_regexp: bool,
    invert_match: bool,
) -> CandidatePredicate:
    regex = _compile_grep_regex(
        pattern,
        case_mode=case_mode,
        fixed_strings=fixed_strings,
        word_regexp=word_regexp,
    )
    if invert_match:
        return CandidatePredicate(
            sql="",
            params={},
            description="no positive content pushdown for invert-match",
            regex=regex,
            required_literals=(),
            approximate_trigrams={},
        )

    case_insensitive = bool(_regex_flags_for_mode(case_mode, pattern) & re.IGNORECASE)
    like_operator = "ILIKE" if case_insensitive else "LIKE"
    regex_operator = "~*" if case_insensitive else "~"

    clauses: list[str] = []
    params: dict[str, Any] = {}
    required_literals: tuple[str, ...]

    if fixed_strings:
        params["fixed_like"] = "%" + _escape_like(pattern) + "%"
        clauses.append(f"content {like_operator} :fixed_like ESCAPE '\\'")
        required_literals = (pattern,)
        description = f"fixed string via {like_operator}"
    else:
        required_literals = required_regex_literals(regex.pattern)
        for idx, literal in enumerate(required_literals):
            key = f"literal_like_{idx}"
            params[key] = "%" + _escape_like(literal) + "%"
            clauses.append(f"content {like_operator} :{key} ESCAPE '\\'")

        pg_pattern = _python_regex_to_postgres(regex.pattern).replace(r"[ \t]", "[[:space:]]")
        if contains_unescaped_anchor(regex.pattern) and not pg_pattern.startswith("(?n)"):
            pg_pattern = "(?n)" + pg_pattern
        params["content_regex"] = pg_pattern
        clauses.append(f"content {regex_operator} :content_regex")
        description = f"regex via {regex_operator}; PostgreSQL pg_trgm extracts candidate trigrams"

    trigram_source = required_literals or tuple(dict.fromkeys(re.findall(r"[A-Za-z0-9_]{3,}", regex.pattern))) or (pattern,)
    trigrams = {literal: approximate_trigrams(literal) for literal in trigram_source}
    return CandidatePredicate(
        sql=" AND ".join(clauses),
        params=params,
        description=description,
        regex=regex,
        required_literals=required_literals,
        approximate_trigrams=trigrams,
    )


In [4]:
def chunk_owner_path(chunk_path: str) -> str:
    return chunk_path.split(CHUNK_MARKER, 1)[0]


def chunk_path_like_for_glob(glob_pattern: str | None) -> str | None:
    if not glob_pattern:
        return None
    owner_like = glob_to_sql_like(glob_pattern)
    if owner_like is None:
        return None
    # Chunk rows are stored as /.vfs/<repo-ish-prefix>/<original-path>/__meta__/chunks/<chunk-id>.
    return re.sub(r"%+", "%", "/.vfs/%/" + owner_like.lstrip("/") + CHUNK_MARKER + "%")


def owner_path_variants(owner_path: str) -> tuple[str, ...]:
    """Return plausible original-path views for exact Python glob filtering."""
    variants = [owner_path]
    parts = owner_path.strip("/").split("/")
    if parts and parts[0] == ".vfs":
        for idx in range(1, len(parts)):
            variants.append("/" + "/".join(parts[idx:]))
    return tuple(dict.fromkeys(variants))


def glob_matches_owner(owner_path: str, glob_pattern: str | None) -> bool:
    if not glob_pattern:
        return True
    compiled = compile_glob(glob_pattern)
    if compiled is None:
        raise ValueError(f"Invalid glob: {glob_pattern!r}")
    return any(compiled.match(variant) is not None for variant in owner_path_variants(owner_path))


def line_matches(content: str, regex: re.Pattern[str], *, invert_match: bool = False) -> list[tuple[int, str]]:
    matches: list[tuple[int, str]] = []
    for line_no, line in enumerate(content.splitlines(), start=1):
        hit = regex.search(line) is not None
        if invert_match:
            hit = not hit
        if hit:
            matches.append((line_no, line))
    return matches


def display_name(owner_path: str) -> str:
    return PurePosixPath(owner_path).name or owner_path


## Inspect the Translation

This cell shows what the harness will ask PostgreSQL to do. The trigram list is an approximation for human inspection; the actual indexed trigrams are generated by PostgreSQL `pg_trgm` for `LIKE`, `ILIKE`, `~`, and `~*`.

In [5]:
predicate = build_content_predicate(
    pattern,
    case_mode=case_mode,
    fixed_strings=fixed_strings,
    word_regexp=word_regexp,
    invert_match=invert_match,
)
glob_like = chunk_path_like_for_glob(glob)

print("content pushdown:", predicate.description)
print("content SQL:", predicate.sql or "<none>")
print("params:")
pprint(predicate.params)
print("required literals:", predicate.required_literals)
print("approximate trigrams:")
pprint(predicate.approximate_trigrams)
print("chunk path LIKE for glob:", glob_like)


content pushdown: regex via ~; PostgreSQL pg_trgm extracts candidate trigrams
content SQL: content ~ :content_regex
params:
{'content_regex': 'Postgres(FileSystem|Backend)'}
required literals: ()
approximate trigrams:
{'Backend': ('ack', 'bac', 'cke', 'end', 'ken'),
 'FileSystem': ('esy', 'fil', 'ile', 'les', 'ste', 'sys', 'tem', 'yst'),
 'Postgres': ('gre', 'ost', 'pos', 'res', 'stg', 'tgr')}
chunk path LIKE for glob: /.vfs/%/%.py/__meta__/chunks/%


In [6]:
def build_candidate_sql(
    predicate: CandidatePredicate,
    *,
    glob_like: str | None,
    limit: int,
    explain: bool = False,
) -> tuple[str, dict[str, Any]]:
    where = [
        "kind = 'chunk'",
        "content IS NOT NULL",
        "deleted_at IS NULL",
    ]
    params = dict(predicate.params)
    if predicate.sql:
        where.append(f"({predicate.sql})")
    if glob_like is not None:
        where.append("path LIKE :glob_like ESCAPE '\\'")
        params["glob_like"] = glob_like
    params["limit"] = limit

    prefix = "EXPLAIN (ANALYZE, BUFFERS)\n" if explain else ""
    sql = f"""
{prefix}SELECT path, line_start, line_end, content
FROM {table_ref()}
WHERE {' AND '.join(where)}
ORDER BY path, line_start
LIMIT :limit
"""
    return sql, params


async def fetch_all(sql: str, params: dict[str, Any]) -> list[dict[str, Any]]:
    async with engine.connect() as conn:
        result = await conn.execute(text(sql), params)
        return [dict(row._mapping) for row in result]


candidate_sql, candidate_params = build_candidate_sql(predicate, glob_like=glob_like, limit=SQL_LIMIT)
print(candidate_sql)
pprint(candidate_params)



SELECT path, line_start, line_end, content
FROM public.vfs_entries
WHERE kind = 'chunk' AND content IS NOT NULL AND deleted_at IS NULL AND (content ~ :content_regex) AND path LIKE :glob_like ESCAPE '\'
ORDER BY path, line_start
LIMIT :limit

{'content_regex': 'Postgres(FileSystem|Backend)',
 'glob_like': '/.vfs/%/%.py/__meta__/chunks/%',
 'limit': 5000}


## Run SQL Candidate Query

Look for `Bitmap Index Scan on ix_vfs_entries_chunk_content_trgm_gin` in the `EXPLAIN` output. If a PostgreSQL regex is rejected because it uses Python-only syntax, make the SQL predicate weaker and rerun; the final Python matcher remains authoritative.

In [7]:
if RUN_EXPLAIN:
    explain_sql, explain_params = build_candidate_sql(predicate, glob_like=glob_like, limit=SQL_LIMIT, explain=True)
    plan_rows = await fetch_all(explain_sql, explain_params)
    print("\n".join(row["QUERY PLAN"] for row in plan_rows))

candidate_rows = await fetch_all(candidate_sql, candidate_params)
print(f"SQL chunk candidates: {len(candidate_rows):,}")


Limit  (cost=997.03..997.09 rows=23 width=829) (actual time=459.315..459.318 rows=33.00 loops=1)
  Buffers: shared hit=4533 read=12200 written=3720
  ->  Sort  (cost=997.03..997.09 rows=23 width=829) (actual time=459.313..459.315 rows=33.00 loops=1)
        Sort Key: path, line_start
        Sort Method: quicksort  Memory: 83kB
        Buffers: shared hit=4533 read=12200 written=3720
        ->  Bitmap Heap Scan on vfs_entries  (cost=301.19..996.51 rows=23 width=829) (actual time=115.203..459.104 rows=33.00 loops=1)
              Recheck Cond: (((content)::text ~ 'Postgres(FileSystem|Backend)'::text) AND ((kind)::text = 'chunk'::text) AND (deleted_at IS NULL))
              Rows Removed by Index Recheck: 362
              Filter: ((path)::text ~~ '/.vfs/%/%.py/__meta__/chunks/%'::text)
              Rows Removed by Filter: 164
              Heap Blocks: exact=324
              Buffers: shared hit=4527 read=12200 written=3720
              ->  Bitmap Index Scan on ix_vfs_entries_chunk_c

## Final Python Filtering and Grouping

The grouping key is the owning file path derived from the chunk path by trimming `/__meta__/chunks/...`. The final match uses the repo's `_compile_grep_regex` result line by line.

In [8]:
def group_verified_matches(
    rows: list[dict[str, Any]],
    *,
    regex: re.Pattern[str],
    glob_pattern: str | None,
    invert_match: bool,
) -> dict[str, list[dict[str, Any]]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        owner = chunk_owner_path(row["path"])
        if not glob_matches_owner(owner, glob_pattern):
            continue
        matches = line_matches(row["content"], regex, invert_match=invert_match)
        if not matches:
            continue
        grouped[owner].append(
            {
                "file_name": display_name(owner),
                "chunk_path": row["path"],
                "line_start": row.get("line_start"),
                "line_end": row.get("line_end"),
                "matches": matches,
                "content": row["content"],
            }
        )
    return dict(sorted(grouped.items()))


verified = group_verified_matches(
    candidate_rows,
    regex=predicate.regex,
    glob_pattern=glob,
    invert_match=invert_match,
)
print(f"Verified files: {len(verified):,}")
print(f"Verified chunks: {sum(len(chunks) for chunks in verified.values()):,}")


Verified files: 9
Verified chunks: 33


In [9]:
def preview_verified(grouped: dict[str, list[dict[str, Any]]], *, limit: int = DISPLAY_LIMIT) -> None:
    shown = 0
    for owner_path, chunks in grouped.items():
        if shown >= limit:
            break
        print(f"\n{owner_path}")
        for chunk in chunks:
            print(f"  chunk lines {chunk['line_start']}..{chunk['line_end']}: {chunk['chunk_path']}")
            for local_line_no, line in chunk["matches"][:8]:
                absolute_line = None
                if chunk["line_start"] is not None:
                    absolute_line = chunk["line_start"] + local_line_no - 1
                line_label = absolute_line if absolute_line is not None else local_line_no
                print(f"    {line_label}: {line[:240]}")
        shown += 1


preview_verified(verified)



/.vfs/grover/scripts/load_git_repos_chunks_postgres_vfs.py
  chunk lines 1..115: /.vfs/grover/scripts/load_git_repos_chunks_postgres_vfs.py/__meta__/chunks/chunk-000000-tok-000000000
    37: from vfs.backends.postgres import PostgresFileSystem
  chunk lines 254..355: /.vfs/grover/scripts/load_git_repos_chunks_postgres_vfs.py/__meta__/chunks/chunk-000003-tok-000000672
    312:     fs = PostgresFileSystem(engine=engine)

/.vfs/grover/scripts/load_repo_into_postgres_vfs.py
  chunk lines 1..80: /.vfs/grover/scripts/load_repo_into_postgres_vfs.py/__meta__/chunks/chunk-000000-tok-000000000
    14: 3. Loads tracked UTF-8 text files from the repo into a mounted PostgresFileSystem.
    33: from vfs.backends.postgres import PostgresFileSystem
  chunk lines 144..242: /.vfs/grover/scripts/load_repo_into_postgres_vfs.py/__meta__/chunks/chunk-000002-tok-000000448
    198:     fs = PostgresFileSystem(engine=engine)

/.vfs/grover/scripts/postgres_repo_cli_probe.py
  chunk lines 1..82: /.vfs/grover/sc

## Reusable One-Call Harness

Use this function to iterate on patterns without rerunning each setup cell manually.

In [10]:
async def live_grep_to_sql(
    pattern: str,
    *,
    case_mode: str = "sensitive",
    fixed_strings: bool = False,
    word_regexp: bool = False,
    invert_match: bool = False,
    glob: str | None = None,
    sql_limit: int = SQL_LIMIT,
    run_explain: bool = True,
) -> dict[str, list[dict[str, Any]]]:
    pred = build_content_predicate(
        pattern,
        case_mode=case_mode,
        fixed_strings=fixed_strings,
        word_regexp=word_regexp,
        invert_match=invert_match,
    )
    path_like = chunk_path_like_for_glob(glob)
    print("content pushdown:", pred.description)
    print("required literals:", pred.required_literals)
    pprint(pred.approximate_trigrams)
    print("chunk path LIKE:", path_like)

    if run_explain:
        explain_sql, explain_params = build_candidate_sql(pred, glob_like=path_like, limit=sql_limit, explain=True)
        plan = await fetch_all(explain_sql, explain_params)
        print("\n".join(row["QUERY PLAN"] for row in plan))

    sql, params = build_candidate_sql(pred, glob_like=path_like, limit=sql_limit)
    rows = await fetch_all(sql, params)
    grouped = group_verified_matches(rows, regex=pred.regex, glob_pattern=glob, invert_match=invert_match)
    print(f"SQL chunk candidates: {len(rows):,}")
    print(f"Verified files: {len(grouped):,}")
    print(f"Verified chunks: {sum(len(chunks) for chunks in grouped.values()):,}")
    return grouped


## Gnarly Benchmark Cases

These are intentionally mixed across the `/Users/claygendron/Git/Repos` corpus: Python async APIs, VFS/Postgres internals, React hooks, config-token names, SQL/index terms, and broad TODO-style code comments. They avoid lookarounds/backrefs so both PostgreSQL trigram regex pushdown and ripgrep can run comparable regexes.

In [11]:
BENCHMARK_CASES = [
    {
        "name": "py_defs_classes_anchored",
        "pattern": r"^[ \t]*(async[ \t]+def|def|class)[ \t]+[A-Za-z_][A-Za-z0-9_]*",
        "glob": "**/*.py",
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "anchored Python API/class declarations",
    },
    {
        "name": "vfs_postgres_types",
        "pattern": r"\b(Postgres(FileSystem|Backend)|DatabaseFileSystem|VirtualFileSystem)\b",
        "glob": "**/*.py",
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "Grover/VFS backend symbols with nested alternation",
    },
    {
        "name": "async_sqlalchemy_postgres",
        "pattern": r"\b(create_async_engine|AsyncSession|asyncpg|postgresql\+asyncpg|sqlalchemy)\b",
        "glob": "**/*.py",
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "async SQLAlchemy/Postgres integration terms",
    },
    {
        "name": "todo_fixme_hack",
        "pattern": r"\b(TODO|FIXME|XXX|HACK|BUG)\b",
        "glob": None,
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "broad code-comment markers across repos",
    },
    {
        "name": "react_hooks_tsx",
        "pattern": r"\b(useEffect|useMemo|useCallback|useLayoutEffect|React\.FC|createRoot)\b",
        "glob": "**/*.tsx",
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "frontend hook/component identifiers",
    },
    {
        "name": "config_env_tokens",
        "pattern": r"\b(OPENAI_API_KEY|ANTHROPIC_API_KEY|DATABASE_URL|SECRET_KEY|TOKEN|AUTH_SECRET)\b",
        "glob": None,
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "environment/config token names, not secret values",
    },
    {
        "name": "sql_trigram_indexes",
        "pattern": r"\b(CREATE[ \t]+(UNIQUE[ \t]+)?INDEX|USING[ \t]+GIN|gin_trgm_ops|pg_trgm|EXPLAIN[ \t]*\([A-Z, \t]+\))",
        "glob": None,
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "DDL, trigram indexes, and explain probes",
    },
    {
        "name": "api_router_models",
        "pattern": r"\b(FastAPI|APIRouter|BaseModel|pydantic|Request|Response)\b",
        "glob": "**/*.py",
        "case_mode": "sensitive",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "Python web/API model surface across app repos",
    },
    {
        "name": "vector_search_terms",
        "pattern": r"\b(pagerank|centrality|embedding|vector(_search|store)?|BM25|semantic[ \t]+search)\b",
        "glob": None,
        "case_mode": "smart",
        "fixed_strings": False,
        "word_regexp": False,
        "notes": "graph/vector/search vocabulary",
    },
    {
        "name": "fixed_postgres_casefold",
        "pattern": "postgres",
        "glob": None,
        "case_mode": "insensitive",
        "fixed_strings": True,
        "word_regexp": False,
        "notes": "fixed-string case-insensitive baseline",
    },
]

len(BENCHMARK_CASES)


10

In [14]:
import statistics
import time


BENCHMARK_SQL_LIMIT = 1_000_000


async def time_live_grep_case(case: dict[str, Any], *, sql_limit: int = BENCHMARK_SQL_LIMIT) -> dict[str, Any]:
    started = time.perf_counter()
    pred = build_content_predicate(
        case["pattern"],
        case_mode=case.get("case_mode", "sensitive"),
        fixed_strings=case.get("fixed_strings", False),
        word_regexp=case.get("word_regexp", False),
        invert_match=case.get("invert_match", False),
    )
    path_like = chunk_path_like_for_glob(case.get("glob"))
    sql, params = build_candidate_sql(pred, glob_like=path_like, limit=sql_limit)
    rows = await fetch_all(sql, params)
    grouped = group_verified_matches(
        rows,
        regex=pred.regex,
        glob_pattern=case.get("glob"),
        invert_match=case.get("invert_match", False),
    )
    elapsed = time.perf_counter() - started
    return {
        "name": case["name"],
        "elapsed_s": elapsed,
        "sql_chunks": len(rows),
        "verified_files": len(grouped),
        "verified_chunks": sum(len(chunks) for chunks in grouped.values()),
        "pattern": case["pattern"],
        "glob": case.get("glob"),
        "pushdown": pred.description,
    }


def print_benchmark_table(rows: list[dict[str, Any]]) -> None:
    header = f"{'case':28} {'best':>8} {'median':>8} {'chunks':>8} {'files':>7} {'verified':>9}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['name'][:28]:28} "
            f"{row['best_s']:8.3f} "
            f"{row['median_s']:8.3f} "
            f"{row['sql_chunks']:8,d} "
            f"{row['verified_files']:7,d} "
            f"{row['verified_chunks']:9,d}"
        )


async def run_live_grep_benchmarks(
    cases: list[dict[str, Any]] = BENCHMARK_CASES,
    *,
    repeats: int = 3,
    warmups: int = 1,
    sql_limit: int = BENCHMARK_SQL_LIMIT,
) -> list[dict[str, Any]]:
    summaries: list[dict[str, Any]] = []
    for case in cases:
        for _ in range(warmups):
            await time_live_grep_case(case, sql_limit=sql_limit)

        runs = [await time_live_grep_case(case, sql_limit=sql_limit) for _ in range(repeats)]
        times = [run["elapsed_s"] for run in runs]
        last = runs[-1]
        summaries.append(
            {
                **last,
                "repeats": repeats,
                "best_s": min(times),
                "median_s": statistics.median(times),
                "all_times_s": times,
            }
        )
        print_benchmark_table(summaries)
        print()
    return summaries


# Run after the DB is warm and reachable:
live_benchmark_results = await run_live_grep_benchmarks(repeats=3, warmups=1)


case                             best   median   chunks   files  verified
-------------------------------------------------------------------------
py_defs_classes_anchored       24.807   25.118  239,724  40,361   239,724

case                             best   median   chunks   files  verified
-------------------------------------------------------------------------
py_defs_classes_anchored       24.807   25.118  239,724  40,361   239,724
vfs_postgres_types              8.323    8.351      226      46       226

case                             best   median   chunks   files  verified
-------------------------------------------------------------------------
py_defs_classes_anchored       24.807   25.118  239,724  40,361   239,724
vfs_postgres_types              8.323    8.351      226      46       226
async_sqlalchemy_postgres       2.105    2.111    4,781   1,463     4,781

case                             best   median   chunks   files  verified
-----------------------------------

## Equivalent Ripgrep Commands

This prints shell commands I can run later against `/Users/claygendron/Git/Repos`. They redirect output to `/tmp` so timing is not dominated by terminal rendering. The ignore globs mirror the database benchmark's intent to search source-like content rather than `.git`, dependency caches, and build output.

In [13]:
import shlex

REPOS_ROOT = "/Users/claygendron/Git/Repos"
RG_EXCLUDE_GLOBS = [
    "!**/.git/**",
    "!**/node_modules/**",
    "!**/.venv/**",
    "!**/dist/**",
    "!**/build/**",
    "!**/target/**",
    "!**/vendor/**",
    "!**/.next/**",
    "!**/__pycache__/**",
]


def rg_command_for_case(case: dict[str, Any], *, root: str = REPOS_ROOT) -> str:
    args = [
        "rg",
        "--line-number",
        "--with-filename",
        "--no-heading",
        "--hidden",
        "--color",
        "never",
    ]
    for exclude_glob in RG_EXCLUDE_GLOBS:
        args.extend(["--glob", exclude_glob])
    if case.get("glob"):
        args.extend(["--glob", case["glob"]])
    if case.get("fixed_strings"):
        args.append("--fixed-strings")
    if case.get("word_regexp"):
        args.append("--word-regexp")
    if case.get("case_mode") == "insensitive":
        args.append("--ignore-case")
    elif case.get("case_mode") == "smart":
        args.append("--smart-case")
    args.extend(["--regexp", case["pattern"], root])
    out = f"/tmp/grover-rg-{case['name']}.out"
    return f"/usr/bin/time -p {shlex.join(args)} > {shlex.quote(out)}"


for case in BENCHMARK_CASES:
    print(f"# {case['name']}: {case['notes']}")
    print(rg_command_for_case(case))
    print()


# py_defs_classes_anchored: anchored Python API/class declarations
/usr/bin/time -p rg --line-number --with-filename --no-heading --hidden --color never --glob '!**/.git/**' --glob '!**/node_modules/**' --glob '!**/.venv/**' --glob '!**/dist/**' --glob '!**/build/**' --glob '!**/target/**' --glob '!**/vendor/**' --glob '!**/.next/**' --glob '!**/__pycache__/**' --glob '**/*.py' --regexp '^[ \t]*(async[ \t]+def|def|class)[ \t]+[A-Za-z_][A-Za-z0-9_]*' /Users/claygendron/Git/Repos > /tmp/grover-rg-py_defs_classes_anchored.out

# vfs_postgres_types: Grover/VFS backend symbols with nested alternation
/usr/bin/time -p rg --line-number --with-filename --no-heading --hidden --color never --glob '!**/.git/**' --glob '!**/node_modules/**' --glob '!**/.venv/**' --glob '!**/dist/**' --glob '!**/build/**' --glob '!**/target/**' --glob '!**/vendor/**' --glob '!**/.next/**' --glob '!**/__pycache__/**' --glob '**/*.py' --regexp '\b(Postgres(FileSystem|Backend)|DatabaseFileSystem|VirtualFileSystem)\b' 